# Task 1 : Set up colab gpu runtime environment

In [ ]:
!pip install segmentation-models-pytorch
!pip install -U git+https://github.com/albumentations-team/albumentations
!pip install --upgrade opencv-contrib-python

^C


# Download Dataset

original author of the dataset :
https://github.com/VikramShenoy97/Human-Segmentation-Dataset


In [ ]:
!git clone https://github.com/parth1620/Human-Segmentation-Dataset-master.git

# Some Common Imports

In [ ]:
import sys
sys.path.append('/content/Human-Segmentation-Dataset-master')

In [ ]:
import torch
import cv2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tqdm import tqdm

import helper

# Task : 2 Setup Configurations

In [ ]:
# TASK 2.1: Import Required Libraries for Configuration Setup

import logging
from dataclasses import dataclass
from typing import Tuple, Optional
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss




In [ ]:
# TASK 2.2: Setup Logging System

def setup_logger() -> logging.Logger:
    """
    Sets up and returns a configured logger.
    
    Why: Centralized logging is essential for consistent output throughout
    the notebook. This replaces print() statements and provides structured logging.
    
    What: Creates a logger instance with a standard formatter that displays
    log messages in normal/default color without any color formatting.
    
    Returns:
        logging.Logger: Configured logger instance ready for use.
        
    Raises:
        None: This function does not raise exceptions.
    """
    logger = logging.getLogger('segmentation_config')
    logger.setLevel(logging.DEBUG)
    
    # Remove existing handlers to avoid duplicates
    if logger.handlers:
        logger.handlers.clear()
    
    # Create console handler
    handler = logging.StreamHandler()
    handler.setLevel(logging.DEBUG)
    
    # Create standard formatter without color formatting
    formatter = logging.Formatter('%(levelname)s: %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    
    return logger

# Initialize logger
logger = setup_logger()



In [ ]:
# TASK 2.3: Define Custom Exceptions for Configuration Errors

class ConfigError(Exception):
    """
    Base exception for configuration-related errors.
    
    Why: Provides specific error handling for configuration validation failures,
    allowing callers to catch and handle configuration errors separately from
    other types of errors.
    
    What: A custom exception class that inherits from Exception, providing
    a clear semantic meaning for configuration-related issues.
    """
    pass


class InvalidDeviceError(ConfigError):
    """
    Exception raised when an invalid device is specified in configuration.
    
    Why: Device validation requires checking CUDA availability, which is a
    specific error condition that should be handled explicitly.
    
    What: Specialized exception for device-related configuration errors,
    such as requesting CUDA when it's not available.
    """
    pass


class InvalidConfigValueError(ConfigError):
    """
    Exception raised when a configuration value is invalid (e.g., negative batch size).
    
    Why: Configuration values must be validated to prevent runtime errors during
    training. Invalid values should be caught early with clear error messages.
    
    What: Specialized exception for invalid configuration parameter values,
    providing clear feedback about what went wrong.
    """
    pass



In [ ]:
# TASK 2.4: Define Type-Safe Configuration Dataclass

@dataclass
class SegmentationConfig:
    """
    Type-safe configuration container for segmentation model training.
    
    Why: Using a dataclass provides type safety, validation, and clear documentation
    of all configuration parameters. This prevents runtime errors from typos or
    invalid values and makes the configuration self-documenting.
    
    What: A dataclass that encapsulates all hyperparameters and model settings
    for the segmentation training pipeline. Configured for grayscale image training
    with a small network (ResNet18) trained from scratch (no pre-trained weights).
    Includes validation logic in __post_init__ to ensure all values are within
    acceptable ranges.
    
    Attributes:
        backbone: Encoder backbone name (e.g., 'resnet18'). ResNet18 is chosen as
            a small, lightweight backbone suitable for training from scratch on grayscale
            images. Smaller networks train faster and require less memory while still
            providing good feature extraction capabilities.
        encoder_weights: Pre-trained weights to use. None means training from scratch,
            which is appropriate for grayscale images where ImageNet pre-trained weights
            (trained on RGB) are not applicable.
        classes: Number of output classes. Set to 1 for binary segmentation
            (can be extended to multi-class).
        activation: Activation function for model output. None means activation
            will be applied in the loss function (sigmoid), not in the model.
        device: Computing device ('cuda' or 'cpu'). CUDA is preferred for GPU acceleration.
        batch_size: Number of samples per training batch. 8 fits standard Colab GPU
            memory (can be increased to 16 if memory allows).
        learning_rate: Initial learning rate for optimizer. 1e-3 is a good starting
            point with potential for reduction during training via scheduler.
        epochs: Maximum number of training epochs.
        early_stop_patience: Number of epochs without improvement before stopping
            training to prevent overfitting.
        in_channels: Number of input image channels (default: 1 for GRAYSCALE).
        scheduler_factor: Factor by which learning rate is reduced (default: 0.5).
        scheduler_patience: Number of epochs to wait before reducing LR (default: 3).
    """
    backbone: str = 'resnet18'
    encoder_weights: Optional[str] = None
    classes: int = 1
    activation: Optional[str] = None
    device: str = 'cuda'
    batch_size: int = 8
    learning_rate: float = 1e-3
    epochs: int = 30
    early_stop_patience: int = 7
    in_channels: int = 1
    scheduler_factor: float = 0.5
    scheduler_patience: int = 3
    
    def __post_init__(self) -> None:
        """
        Validate configuration values after initialization.
        
        Why: Ensures all configuration values are valid before they are used,
        catching errors early and providing clear feedback.
        
        What: Performs validation checks on all configuration parameters:
        - Validates device availability (CUDA if requested)
        - Ensures positive values for batch size, learning rate, epochs, etc.
        - Validates that early stop patience is positive
        
        Raises:
            InvalidDeviceError: If CUDA is requested but not available.
            InvalidConfigValueError: If any configuration value is invalid.
        """
        self._validate_device()
        self._validate_positive_values()
        self._validate_learning_rate()
    
    def _validate_device(self) -> None:
        """Validate that the requested device is available."""
        if self.device == 'cuda' and not torch.cuda.is_available():
            raise InvalidDeviceError(
                "CUDA device requested but not available. "
                "Please set device='cpu' or ensure CUDA is properly configured."
            )
    
    def _validate_positive_values(self) -> None:
        """Validate that all positive-value parameters are indeed positive."""
        if self.batch_size <= 0:
            raise InvalidConfigValueError(f"batch_size must be positive, got {self.batch_size}")
        if self.epochs <= 0:
            raise InvalidConfigValueError(f"epochs must be positive, got {self.epochs}")
        if self.early_stop_patience <= 0:
            raise InvalidConfigValueError(
                f"early_stop_patience must be positive, got {self.early_stop_patience}"
            )
        if self.classes <= 0:
            raise InvalidConfigValueError(f"classes must be positive, got {self.classes}")
        if self.in_channels <= 0:
            raise InvalidConfigValueError(f"in_channels must be positive, got {self.in_channels}")
        if self.scheduler_factor <= 0:
            raise InvalidConfigValueError(
                f"scheduler_factor must be positive, got {self.scheduler_factor}"
            )
        if self.scheduler_patience <= 0:
            raise InvalidConfigValueError(
                f"scheduler_patience must be positive, got {self.scheduler_patience}"
            )
    
    def _validate_learning_rate(self) -> None:
        """Validate that learning rate is within acceptable range."""
        if self.learning_rate <= 0:
            raise InvalidConfigValueError(
                f"learning_rate must be positive, got {self.learning_rate}"
            )
        if self.learning_rate > 1.0:
            raise InvalidConfigValueError(
                f"learning_rate seems unusually high: {self.learning_rate}. "
                "Typical values are between 1e-5 and 1e-1."
            )



In [ ]:
# TASK 2.5: Create Segmentation Model Factory Function

def create_segmentation_model(config: SegmentationConfig) -> torch.nn.Module:
    """
    Creates and returns a UNet segmentation model based on configuration.
    
    Why: UNet is an excellent architecture for segmentation tasks and is easy to train.
    Using a small backbone like ResNet18 allows for efficient training from scratch
    on grayscale images. Since we're training without pre-trained weights, a smaller
    network is more efficient and reduces the risk of overfitting on smaller datasets.
    
    What: Instantiates a UNet model from segmentation_models_pytorch library with
    the specified encoder backbone, encoder weights (None for training from scratch),
    and output configuration. The model is then moved to the specified device (CPU or CUDA).
    
    Args:
        config: SegmentationConfig instance containing all model hyperparameters
            including backbone name, encoder weights, number of classes, activation,
            and target device.
    
    Returns:
        torch.nn.Module: Initialized UNet model ready for training, moved to the
            specified device.
    
    Raises:
        RuntimeError: If model creation fails or device transfer fails.
    """
    logger.debug(f"Creating UNet model with backbone: {config.backbone}")
    
    model = smp.Unet(
        encoder_name=config.backbone,
        encoder_weights=config.encoder_weights,
        in_channels=config.in_channels,
        classes=config.classes,
        activation=config.activation
    )
    
    device = torch.device(config.device)
    model = model.to(device)
    
    logger.info(f"Model created successfully and moved to {config.device}")
    return model



In [ ]:
# TASK 2.6: Create Training Components Factory Function

def create_training_components(
    model: torch.nn.Module,
    config: SegmentationConfig
) -> Tuple[torch.nn.Module, torch.optim.Optimizer, torch.optim.lr_scheduler.ReduceLROnPlateau]:
    """
    Creates and returns loss function, optimizer, and learning rate scheduler.
    
    Why: Centralizing the creation of training components ensures consistency and
    makes it easy to modify training setup. Adam optimizer enables fast convergence
    and good learning. DiceLoss is specifically designed for segmentation accuracy.
    The scheduler allows automatic learning rate reduction if loss doesn't improve,
    preventing wasted training time.
    
    What: Creates three essential training components:
    1. DiceLoss for binary segmentation - provides accurate segmentation class prediction
    2. Adam optimizer with specified learning rate - enables fast convergence
    3. ReduceLROnPlateau scheduler - reduces learning rate if loss plateaus
    
    Args:
        model: The PyTorch model whose parameters will be optimized.
        config: SegmentationConfig instance containing optimizer and scheduler
            hyperparameters (learning_rate, scheduler_factor, scheduler_patience).
    
    Returns:
        Tuple containing:
            - loss_fn: DiceLoss instance configured for binary segmentation
            - optimizer: Adam optimizer configured with model parameters and learning rate
            - scheduler: ReduceLROnPlateau scheduler configured to reduce LR on plateau
    
    Raises:
        ValueError: If model has no parameters to optimize.
    """
    logger.debug("Creating training components: loss, optimizer, scheduler")
    
    # Create DiceLoss for binary segmentation
    # DiceLoss is specifically designed for segmentation tasks and provides
    # accurate class prediction. Can be combined with BCE-DiceLoss for stability.
    loss_fn = DiceLoss(mode='binary')
    logger.debug("DiceLoss created with binary mode")
    
    # Create Adam optimizer
    # Adam optimizer enables fast convergence and good learning performance
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    logger.debug(f"Adam optimizer created with learning_rate={config.learning_rate}")
    
    # Create learning rate scheduler
    # Reduces learning rate if loss doesn't improve for several epochs,
    # preventing wasted training time and potentially finding better minima
    # Note: verbose parameter is not supported in newer PyTorch versions.
    # Learning rate changes will be logged manually during training.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=config.scheduler_factor,
        patience=config.scheduler_patience
    )
    logger.debug(
        f"ReduceLROnPlateau scheduler created with factor={config.scheduler_factor}, "
        f"patience={config.scheduler_patience}"
    )
    
    logger.info("All training components created successfully")
    return loss_fn, optimizer, scheduler



In [ ]:
# TASK 2.7: Configuration Logging Functions

def _log_configuration_header(logger_instance: logging.Logger) -> None:
    """
    Log configuration section header.
    
    Why: Encapsulates header logging to avoid duplication and maintain
    consistent formatting across configuration logging functions.
    
    What: Logs a formatted header section for configuration output.
    
    Args:
        logger_instance: Logger instance to use for output.
    
    Returns:
        None
    
    Raises:
        None: This function does not raise exceptions.
    """
    logger_instance.info("=" * 50)
    logger_instance.info("Segmentation Model Configuration")
    logger_instance.info("=" * 50)


def _log_model_architecture(config: SegmentationConfig, logger_instance: logging.Logger) -> None:
    """
    Log model architecture parameters.
    
    Why: Separates architecture parameter logging to maintain single
    responsibility and improve code organization.
    
    What: Logs backbone, encoder weights, input channels, output classes,
    and activation function from the configuration.
    
    Args:
        config: SegmentationConfig instance containing architecture parameters.
        logger_instance: Logger instance to use for output.
    
    Returns:
        None
    
    Raises:
        None: This function does not raise exceptions.
    """
    logger_instance.info(f"Backbone: {config.backbone}")
    encoder_weights_display = (
        "None (training from scratch)" 
        if config.encoder_weights is None 
        else config.encoder_weights
    )
    logger_instance.info(f"Encoder Weights: {encoder_weights_display}")
    channel_type = 'GRAYSCALE' if config.in_channels == 1 else 'RGB'
    logger_instance.info(f"Input Channels: {config.in_channels} ({channel_type})")
    logger_instance.info(f"Output Classes: {config.classes}")
    logger_instance.info(f"Activation: {config.activation}")


def _log_training_hyperparameters(config: SegmentationConfig, logger_instance: logging.Logger) -> None:
    """
    Log training hyperparameters.
    
    Why: Separates hyperparameter logging to maintain single responsibility
    and improve code organization.
    
    What: Logs device, batch size, learning rate, epochs, and early stop
    patience from the configuration.
    
    Args:
        config: SegmentationConfig instance containing hyperparameters.
        logger_instance: Logger instance to use for output.
    
    Returns:
        None
    
    Raises:
        None: This function does not raise exceptions.
    """
    logger_instance.info(f"Device: {config.device}")
    logger_instance.info(f"Batch Size: {config.batch_size}")
    logger_instance.info(f"Learning Rate: {config.learning_rate}")
    logger_instance.info(f"Epochs: {config.epochs}")
    logger_instance.info(f"Early Stop Patience: {config.early_stop_patience}")


def _log_scheduler_parameters(config: SegmentationConfig, logger_instance: logging.Logger) -> None:
    """
    Log scheduler parameters.
    
    Why: Separates scheduler parameter logging to maintain single responsibility
    and improve code organization.
    
    What: Logs scheduler factor and patience from the configuration.
    
    Args:
        config: SegmentationConfig instance containing scheduler parameters.
        logger_instance: Logger instance to use for output.
    
    Returns:
        None
    
    Raises:
        None: This function does not raise exceptions.
    """
    logger_instance.info(f"Scheduler Factor: {config.scheduler_factor}")
    logger_instance.info(f"Scheduler Patience: {config.scheduler_patience}")


def log_configuration(config: SegmentationConfig, logger_instance: logging.Logger) -> None:
    """
    Log all configuration parameters using helper functions.
    
    Why: Provides a single entry point for configuration logging, orchestrating
    multiple helper functions to maintain code organization and readability.
    
    What: Logs configuration header, model architecture, training hyperparameters,
    scheduler parameters, and design rationale using dedicated helper functions.
    
    Args:
        config: SegmentationConfig instance to log.
        logger_instance: Logger instance to use for output.
    
    Returns:
        None
    
    Raises:
        None: This function does not raise exceptions.
    """
    _log_configuration_header(logger_instance)
    _log_model_architecture(config, logger_instance)
    _log_training_hyperparameters(config, logger_instance)
    _log_scheduler_parameters(config, logger_instance)
    logger_instance.info("=" * 50)
    logger_instance.debug(
        "Design rationale: UNet+ResNet18 provides efficient training from scratch. "
        "DiceLoss is precise for segmentation tasks. "
        "Optimizer and early stopping prevent overfitting."
    )



In [ ]:
# TASK 2.8: Main Execution - Create Configuration and Initialize Components

# Create configuration instance with default values
# All values can be customized by passing arguments to SegmentationConfig()
config = SegmentationConfig()

# Create the segmentation model
model = create_segmentation_model(config)

# Create training components (loss, optimizer, scheduler)
loss_fn, optimizer, scheduler = create_training_components(model, config)

# Log the complete configuration
log_configuration(config, logger)

logger.info("TASK 2 completed: All components initialized and ready for training")



# Task 2.1: Load Image Database


In [ ]:
# TASK 2.1: Import Required Libraries for Dataset Loading

import os
from pathlib import Path
from typing import List, Tuple
from PIL import Image
import numpy as np

logger.info("Imported libraries for dataset loading")


In [ ]:
# TASK 2.1: Download Dataset (if not already downloaded)

# The dataset should already be downloaded from Task 1
# If needed, uncomment the line below:
# !git clone https://github.com/parth1620/Human-Segmentation-Dataset-master.git

dataset_path = '/content/Human-Segmentation-Dataset-master'
logger.info(f"Dataset path: {dataset_path}")


In [ ]:
CSV_FILE = '/content/Human-Segmentation-Dataset-master/train.csv'
DTAT_DIR = '/content/'
df = pd.read_csv(CSV_FILE)

row = df.iloc[4]

image_path = row.images
mask_path = row.masks

# Load image as grayscale (single channel)
# cv2.IMREAD_GRAYSCALE loads the image directly as grayscale
image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# Load mask as grayscale
mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

In [ ]:
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,5))

ax1.set_title('IMAGE (GRAYSCALE)')
ax1.imshow(image, cmap='gray')

ax2.set_title('GROUND TRUTH')
ax2.imshow(mask, cmap='gray')

# Task 3 : Augmentation Functions

In [ ]:
# TASK 3.1: Import Required Libraries for Augmentation

import albumentations as A
from albumentations.pytorch import ToTensorV2
from typing import Dict, Any, Tuple

logger.info("Imported Albumentations library for data augmentation")


In [ ]:
# TASK 3.2: Create Training Augmentation Pipeline

def create_train_augmentation() -> A.Compose:
    """
    Creates and returns a training augmentation pipeline using Albumentations.
    
    Why: Data augmentation is crucial for small datasets to increase effective
    dataset size and improve model generalization. Using high probability values
    (0.8-0.9) ensures that most samples are augmented, maximizing the benefit
    for small datasets. All augmentations are applied synchronously to both
    image and mask, which is essential for segmentation tasks.
    
    What: Creates a composition of augmentations including ONLY intensity
    transformations that do not affect image/mask dimensions:
    - Intensity transformations: Brightness, Contrast, Gamma Correction
    
    Note: Geometric transformations (Rotation, Flip, Shift, Scale) have been
    removed to avoid misalignment issues when image and mask have different sizes.
    Only intensity augmentations are used as they do not require size matching.
    
    All augmentations use high probability (0.85) to maximize augmentation
    frequency for small datasets. The pipeline is configured to work with
    grayscale images and binary segmentation masks.
    
    Important: 
    - Only intensity augmentations are used (no geometric transformations)
    - These augmentations do not affect image/mask dimensions, so size mismatches
      are not a problem
    - Intensity augmentations only affect the image, not the mask
    
    Returns:
        A.Compose: Albumentations composition pipeline ready for training data.
        
    Raises:
        None: This function does not raise exceptions.
    """
    logger.debug("Creating training augmentation pipeline")
    
    # High probability for small dataset - we want most samples to be augmented
    # Using 0.85 probability ensures ~85% of samples get each augmentation
    high_prob = 0.85
    
    train_transform = A.Compose([
        # Intensity Augmentations Only (No Geometric Transformations)
        # These augmentations only affect the image, not the mask, and do not
        # require matching image/mask dimensions.
        
        # Brightness and Contrast Augmentation
        A.OneOf([
            A.RandomBrightnessContrast(
                brightness_limit=0.2,  # Adjust brightness by ±20%
                contrast_limit=0.2,  # Adjust contrast by ±20%
                p=1.0  # Always apply when selected
            ),
            A.NoOp()  # Do nothing - this ensures mask is not affected
        ], p=high_prob),
        
        # Gamma Correction Augmentation
        A.OneOf([
            A.RandomGamma(
                gamma_limit=(80, 120),  # Gamma correction between 0.8 and 1.2
                p=1.0  # Always apply when selected
            ),
            A.NoOp()  # Do nothing - this ensures mask is not affected
        ], p=high_prob),
        
    ], is_check_shapes=False)  # Safe: intensity transforms don't change dimensions
    
    logger.info(
        f"Training augmentation pipeline created with high probability ({high_prob}) "
        "for maximum augmentation on small dataset"
    )
    logger.debug(
        "Augmentations included: BrightnessContrast, GammaCorrection (intensity only)"
    )
    logger.debug(
        "Geometric transformations (Rotation, Flip, Shift, Scale) removed to avoid "
        "size mismatch issues"
    )
    
    return train_transform


In [ ]:
# TASK 3.3: Create Validation Augmentation Pipeline

def create_val_augmentation() -> A.Compose:
    """
    Creates and returns a validation augmentation pipeline.
    
    Why: Validation data should not be augmented to ensure consistent evaluation.
    However, we still need a pipeline structure for consistency with training
    data loading. This pipeline applies Resize to match training pipeline dimensions
    but no other augmentations.
    
    What: Creates a minimal composition for validation data with no augmentations.
    This maintains the same interface as the training pipeline for code consistency.
    Validation data should not be augmented to ensure consistent evaluation.
    
    Returns:
        A.Compose: Albumentations composition pipeline for validation (resize only).
        
    Raises:
        None: This function does not raise exceptions.
    """
    logger.debug("Creating validation augmentation pipeline (no augmentations)")
    
    val_transform = A.Compose([
        # No augmentations for validation - return data as-is
        # This ensures consistent evaluation without any transformations
    ], is_check_shapes=False)  # Safe: no transforms applied
    
    logger.info("Validation augmentation pipeline created (identity transform)")
    
    return val_transform


In [ ]:
# TASK 3.4: Main Execution - Create Augmentation Pipelines

# Create training augmentation pipeline with high probability for small dataset
train_augmentation = create_train_augmentation()

# Create validation augmentation pipeline (no augmentations)
val_augmentation = create_val_augmentation()

logger.info("TASK 3 completed: Augmentation pipelines created and ready for use")


In [ ]:
# TASK 3.5: Helper Functions for Augmentation Visualization

def load_image_and_mask(sample_idx: int, dataframe: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, bool]:
    """
    Load image and mask for a given sample index.
    
    Why: Centralizes image/mask loading logic to avoid duplication and provide
    consistent error handling across visualization code.
    
    What: Loads image and mask from dataframe paths, validates they were loaded
    successfully, and returns them along with a success flag.
    
    Args:
        sample_idx: Index of the sample to load from dataframe.
        dataframe: DataFrame containing image and mask paths.
    
    Returns:
        Tuple containing:
            - original_image: Loaded image array or None if failed
            - original_mask: Loaded mask array or None if failed
            - success: Boolean indicating if loading was successful
    
    Raises:
        None: Returns None values and False flag instead of raising exceptions.
    """
    row = dataframe.iloc[sample_idx]
    image_path = row.images
    mask_path = row.masks
    
    original_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    original_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    if original_image is None or original_mask is None:
        return None, None, False
    
    return original_image, original_mask, True


def normalize_mask_format(mask: np.ndarray) -> np.ndarray:
    """
    Normalize mask to uint8 format with 0-255 range.
    
    Why: Ensures consistent mask format across the visualization pipeline,
    handling both 0-1 and 0-255 input ranges.
    
    What: Converts mask to uint8 format, scaling from 0-1 range if needed,
    or ensuring proper type if already in 0-255 range.
    
    Args:
        mask: Mask array in either 0-1 or 0-255 range.
    
    Returns:
        np.ndarray: Mask in uint8 format with 0-255 range.
    
    Raises:
        None: This function does not raise exceptions.
    """
    if mask.max() <= 1:
        return (mask > 0.5).astype(np.uint8) * 255
    return mask.astype(np.uint8)


def _display_single_image(ax: Any, image: np.ndarray, title: str) -> None:
    """
    Display a single image in a matplotlib axes.
    
    Why: Encapsulates image display logic to avoid code duplication
    in visualization functions.
    
    What: Displays an image with specified title and removes axes.
    
    Args:
        ax: Matplotlib axes to display the image in.
        image: Image array to display.
        title: Title text for the image.
    
    Returns:
        None
    
    Raises:
        None: This function does not raise exceptions.
    """
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')


def display_sample(
    axes: np.ndarray,
    row_idx: int,
    sample_idx: int,
    original_image: np.ndarray,
    original_mask: np.ndarray,
    augmented_image: np.ndarray,
    augmented_mask: np.ndarray
) -> None:
    """
    Display a single sample's images and masks in the visualization grid.
    
    Why: Encapsulates the display logic for a single sample to avoid code
    duplication and improve readability of the main visualization loop.
    
    What: Displays original image, original mask, augmented image, and
    augmented mask in the specified row of the axes grid using helper functions.
    
    Args:
        axes: 2D array of matplotlib axes for the visualization grid.
        row_idx: Row index in the axes grid (0-4).
        sample_idx: Sample index for title display (1-5).
        original_image: Original image array to display.
        original_mask: Original mask array to display.
        augmented_image: Augmented image array to display.
        augmented_mask: Augmented mask array to display.
    
    Returns:
        None
    
    Raises:
        None: This function does not raise exceptions.
    """
    sample_num = sample_idx + 1
    _display_single_image(axes[row_idx, 0], original_image, f'Sample {sample_num}: Original Image')
    _display_single_image(axes[row_idx, 1], original_mask, f'Sample {sample_num}: Original Mask')
    _display_single_image(axes[row_idx, 2], augmented_image, f'Sample {sample_num}: Augmented Image')
    _display_single_image(axes[row_idx, 3], augmented_mask, f'Sample {sample_num}: Augmented Mask')


# TASK 3.6: Main Execution - Visualize Augmentation Results

sample_indices = [0, 1, 2, 3, 4]
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
fig.suptitle('Augmentation Visualization: Original vs Augmented Images and Masks', 
             fontsize=16, fontweight='bold')

for idx, sample_idx in enumerate(sample_indices):
    original_image, original_mask, success = load_image_and_mask(sample_idx, df)
    
    if not success:
        logger.warning(f"Failed to load image or mask for sample {sample_idx}")
        continue
    
    augmented = train_augmentation(image=original_image, mask=original_mask)
    augmented_image = augmented['image']
    augmented_mask = normalize_mask_format(augmented['mask'])
    
    display_sample(axes, idx, sample_idx, original_image, original_mask, 
                   augmented_image, augmented_mask)

plt.tight_layout()
plt.show()
logger.info("Augmentation visualization completed for 5 samples")


albumentation documentation : https://albumentations.ai/docs/

# Task 4 : Create Custom Dataset

In [ ]:
# TASK 4.1: Log Dataset Sizes

logger.info(f"Size of Trainset: {len(trainset)}")
logger.info(f"Size of Validset: {len(validset)}")

# Task 5 : Load dataset into batches

# Task 6 : Create Segmentation Model

segmentation_models_pytorch documentation : https://smp.readthedocs.io/en/latest/

# Task 7 : Create Train and Validation Function

# Task 8 : Train Model

# Task 9 : Inference